# Equivalent noise for majority vote



In [ ]:
import stim


base_circuit = stim.Circuit(
"""
# initialize data qubits and pairs
RX 0 1
R 10 11
CX 0 10 1 11
MXX 0 1
##### first
Z_ERROR(0.001) 0 1
RX 2
R 3
CX 2 3
PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 3
Z_ERROR(0.001) 0 1
CX 2 0 3 1
MX(0.001) 2 3
DETECTOR(0) rec[-1] rec[-2] rec[-3]
# MEASURE PAULI ON THE DATA QUBITS
MXX 0 10
DETECTOR(1, 1) rec[-1]
MXX 1 11
DETECTOR(2, 1) rec[-1]
MZZ 0 10
DETECTOR(3, 1) rec[-1]
MZZ 1 11
DETECTOR(4, 1) rec[-1]
# OBSERVABLE_INCLUDE(0) X0 X10
# OBSERVABLE_INCLUDE(1) Z0 Z10
# OBSERVABLE_INCLUDE(2) X1
# OBSERVABLE_INCLUDE(3) Z1
"""
)


base_circuit.detector_error_model(allow_gauge_detectors=True)
# base_circuit.explain_detector_error_model_errors()


stim.DetectorErrorModel('''
    error(0.0119678) D0
    error(0.001998) D0 D1
    error(0.001998) D0 D2
    error(0.00050025) D0 D4
    error(0.5) D3 D4
    error(0.00050025) D4
    detector(0) D0
    detector(1, 1) D1
    detector(2, 1) D2
    detector(3, 1) D3
    detector(4, 1) D4
''')

In [ ]:
# class noise_source:
#     def __init__(self, p, symplectic: List[bool], measurement: List[bool]):
#         self.p = p
#         self.symplectic = symplectic
#         self.measurement = measurement

# ##### first
# RX 2
# R 3
# CX 2 3
# PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 3
# Z_ERROR(0.001) 0
# Z_ERROR(0.001) 1
# CX 2 0 3 1
# X_ERROR(0.001) 2 
# X_ERROR(0.001) 3
# ##### second
# Z_ERROR(0.001) 0
# Z_ERROR(0.001) 1
# RX 4
# R 5
# CX 4 5
# PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 5
# Z_ERROR(0.001) 0
# Z_ERROR(0.001) 1
# CX 4 0 5 1
# X_ERROR(0.001) 4
# X_ERROR(0.001) 5
# ##### third
# Z_ERROR(0.001) 0 
# Z_ERROR(0.001) 1
# RX 6
# R 7
# CX 6 7
# PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 7
# Z_ERROR(0.001) 0 
# Z_ERROR(0.001) 1
# CX 6 0 7 1
# X_ERROR(0.001) 6
# X_ERROR(0.001) 7
# """

In [26]:
base_circuit = stim.Circuit(
"""
# initialize data qubits and pairs
RX 0 1
R 10 11
CX 0 10 1 11
MXX 0 1
##### first
Z_ERROR(0.001) 0 1
RX 2
R 3
CX 2 3
PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 3
Z_ERROR(0.001) 0 1
CX 2 0 3 1
MX(0.001) 2 3
DETECTOR(0) rec[-1] rec[-2]
##### second
Z_ERROR(0.001) 0 1
RX 4
R 5
CX 4 5
PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 5
Z_ERROR(0.001) 0 1
CX 4 0 5 1
MX(0.001) 4 5
DETECTOR(1) rec[-1] rec[-2]
##### third
Z_ERROR(0.001) 0 1
RX 6
R 7
CX 6 7
PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 7
Z_ERROR(0.001) 0 1
CX 6 0 7 1
MX(0.001) 6 7
DETECTOR(2) rec[-1] rec[-2]
# MEASURE PAULI ON THE DATA QUBITS
MXX 0 10
DETECTOR(1, 1) rec[-1]
MXX 1 11
DETECTOR(2, 1) rec[-1]
MZZ 0 10
DETECTOR(3, 1) rec[-1]
MZZ 1 11
DETECTOR(4, 1) rec[-1]
"""
)

base_circuit.detector_error_model(allow_gauge_detectors=True)


stim.DetectorErrorModel('''
    error(0.0119678) D0
    error(0.5) D0 D1 D2
    error(0.001998) D0 D1 D2 D3
    error(0.001998) D0 D1 D2 D4
    error(0.00050025) D0 D5
    error(0.0119678) D1
    error(0.001998) D1 D2 D3
    error(0.001998) D1 D2 D4
    error(0.00050025) D1 D5
    error(0.0119678) D2
    error(0.001998) D2 D3
    error(0.001998) D2 D4
    error(0.00050025) D2 D6
    error(0.001) D5
    error(0.5) D5 D6
    error(0.00050025) D6
    detector(0) D0
    detector(1) D1
    detector(2) D2
    detector(1, 1) D3
    detector(2, 1) D4
    detector(3, 1) D5
    detector(4, 1) D6
''')

Look at the dem above. we are going to turn in into an equivalent noise model for a repeated measurement 3 times.
The mapping rules are:
D0 - first measurement
D1 - second measrement
D2 - third measurement
D3 - X0
D4 - X1
D5 - Z0
D6 - Z1

equivalent error mechains has:
p: probability for error.
M: measurement error (at least two physical measurement failed)
Xi: X error on the i'th qubit
Zi: Z error on the i'th qubit

do:
1. Go over the dem and discard all the Gauge lines (probability 0.5 and only detectors are D3 D4 D5 D6).
2. Go over every combination of one or two error mechanichms and:
    - Xor the detectors so we will get an equivalent error mechanism.
    - probability is the multiplication of both (maybe we need to also take into account (1-p) of all the other probabilities, should we? or just assume other order won't change.)
    - write which detectors are flipped.
    - calculate equivalent error: 
        - D0, D1 and D2 are majority voted. meaning that we write M error just if at least two have flipped.
        - X0, X1, Z0, Z1 are just xored so this happens already in the previous step.
    - push the equivalent error mechanism to a list.


Some of the building blocks are implemented [here](https://github.com/shohamjac/injection-with-erasure-qubits/blob/407a678aafd693b0380246c843c28a67d1fcfc99/Combination_searching.ipynb#L22).
